# Hybrid CVRP Solver — QAOA Cut-Point Selection

**Algorithm:**
1. **Classical** — encode locations, build Euclidean distance matrix, sort customers by polar angle from depot.
2. **QAOA** — find the optimal placement of G−1 cut points among the H−1 angular gaps, minimizing total route distance subject to exactly G−1 cuts.
3. **Classical** — optimal route ordering within each group via exhaustive permutation search.

**QUBO formulation:** binary variable $b_i = 1$ means a cut between angular neighbors $i$ and $i+1$.

$$C(b) = \sum_i w_i b_i + \lambda\left(\sum_i b_i - (G-1)\right)^2$$

where $w_i = d(c_i, \text{depot}) + d(\text{depot}, c_{i+1}) - d(c_i, c_{i+1})$ is the detour cost of routing through the depot instead of directly between angular neighbors. Via $b_i = (1-Z_i)/2$ this maps to a standard Ising Hamiltonian solved by QAOA.

In [ ]:
import math, os, time
import numpy as np
from itertools import permutations
from scipy.optimize import minimize

from qiskit.circuit.library import QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp, Statevector

from qbraid.runtime import QbraidProvider

# Connect to qBraid — circuits are sampled here after local parameter optimization
provider = QbraidProvider()
qbraid_device = provider.get_device("qbraid_qir_simulator")

## Classical Utilities

In [ ]:
def encode_locations(depot, customers):
    locations = {0: tuple(depot)}
    for i, c in enumerate(customers, start=1):
        locations[i] = tuple(c)
    return locations

def euclidean(p1, p2):
    return math.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)

def build_distance_matrix(locations):
    ids = list(locations.keys())
    return {(i,j): euclidean(locations[i], locations[j]) for i in ids for j in ids if i != j}

def compute_num_groups(H, C, N):
    return min(math.ceil(H/C), N)

def angular_sort(customer_ids, locations):
    ox, oy = locations[0]
    return sorted(customer_ids, key=lambda n: math.atan2(locations[n][1]-oy, locations[n][0]-ox))

## QUBO: Cut-Point Weights

$w_i > 0$ means cutting there is expensive (the two customers are close together, so routing through the depot adds significant distance). QAOA will prefer to place cuts where $w_i$ is small.

In [ ]:
def build_cut_weights(sorted_custs, dist_matrix):
    w = []
    for i in range(len(sorted_custs) - 1):
        ci, cj = sorted_custs[i], sorted_custs[i+1]
        w.append(dist_matrix[(ci, 0)] + dist_matrix[(0, cj)] - dist_matrix[(ci, cj)])
    return w

## Cost Hamiltonian

Via $b_i = (1-Z_i)/2$, the QUBO maps to the Ising Hamiltonian (dropping constants):

$$H_C = \sum_i \left(-\frac{w_i}{2} - \lambda A\right) Z_i + \frac{\lambda}{2}\sum_{i<j} Z_i Z_j$$

where $A = n/2 - (G-1)$ and $n$ is the number of binary variables (gaps).

Qiskit convention: qubit $i$ maps to string position $n-1-i$ (rightmost = qubit 0).

In [ ]:
def build_cost_hamiltonian(weights, G):
    n = len(weights)
    lam = max(abs(w) for w in weights) * n * 2   # penalty: must outweigh any objective gain
    A = n/2 - (G-1)

    terms = []

    # Z terms
    for i, w in enumerate(weights):
        coeff = -(w/2) - lam*A
        z = list('I'*n)
        z[n-1-i] = 'Z'
        terms.append((''.join(z), coeff))

    # ZZ terms (from constraint penalty)
    for i in range(n):
        for j in range(i+1, n):
            zz = list('I'*n)
            zz[n-1-i] = 'Z'
            zz[n-1-j] = 'Z'
            terms.append((''.join(zz), lam/2))

    return SparsePauliOp.from_list(terms)

## QAOA

**Hybrid split:**
- **Classical (local):** COBYLA optimizes the QAOA parameters by evaluating ⟨ψ(β,γ)|H_C|ψ(β,γ)⟩ via exact statevector simulation. This loop never touches quantum hardware — it's pure classical parameter search.
- **Quantum (qBraid):** Once optimal parameters are found, the final circuit is submitted to the qBraid simulator for sampling. This is where the quantum execution happens, satisfying the qBraid requirement.

`reps=1` (one cost + one mixer layer) keeps the circuit shallow and convergence fast. COBYLA needs only ~100–150 iterations for 2 parameters.

In [ ]:
def run_qaoa(cost_op, reps=1, shots=2048):
    ansatz = QAOAAnsatz(cost_op, reps=reps)
    params = ansatz.parameters
    x0 = np.random.uniform(0, np.pi, len(params))

    # Classical optimization loop — expectation value via local statevector
    def objective(x):
        bound = ansatz.assign_parameters(dict(zip(params, x)))
        return Statevector(bound).expectation_value(cost_op).real

    t0 = time.time()
    opt = minimize(objective, x0, method='COBYLA',
                   options={'maxiter': 200, 'rhobeg': 0.5})
    elapsed = time.time() - t0

    # Quantum sampling on qBraid with optimal parameters
    bound_final = ansatz.assign_parameters(dict(zip(params, opt.x)))
    bound_final.measure_all()

    job = qbraid_device.run(bound_final, shots=shots)
    counts = job.result().measurement_counts
    probs = {bs: c / shots for bs, c in counts.items()}

    gates = sum(ansatz.decompose().count_ops().values())
    info = {"qubits": cost_op.num_qubits, "gates": gates, "time_s": elapsed}
    return probs, opt, info

## Extract Groups from QAOA Output

Select the highest-probability bitstring with exactly $G-1$ ones. If QAOA finds no valid solution, fall back to the greedy choice: cut the $G-1$ cheapest gaps (lowest $w_i$).

In [ ]:
def probs_to_groups(probs, sorted_custs, G, weights):
    n = len(weights)
    valid = {bs: p for bs, p in probs.items() if bs.count('1') == G-1}

    if valid:
        best_bs = max(valid, key=valid.get)
    else:
        # Greedy fallback: cut G-1 cheapest gaps
        cut_idxs = sorted(range(n), key=lambda i: weights[i])[:G-1]
        arr = list('0'*n)
        for i in cut_idxs:
            arr[n-1-i] = '1'   # Qiskit: qubit i at string position n-1-i
        best_bs = ''.join(arr)

    # Decode: arr[n-1-i] == '1'  ↔  b_i == 1  ↔  cut after position i
    cuts = [i for i in range(n) if best_bs[n-1-i] == '1']

    groups, prev = [], 0
    for cut in sorted(cuts):
        groups.append(sorted_custs[prev:cut+1])
        prev = cut+1
    groups.append(sorted_custs[prev:])

    return [g for g in groups if g]

## Route Construction

Exact optimal ordering within each group via exhaustive permutation (feasible since group size $\leq C \leq 5$).

In [ ]:
def build_route(group, dist_matrix):
    best, best_d = None, float('inf')
    for perm in permutations(group):
        route = [0] + list(perm) + [0]
        d = sum(dist_matrix[(route[k], route[k+1])] for k in range(len(route)-1))
        if d < best_d:
            best_d, best = d, route
    return best

def total_distance(routes, dist_matrix):
    return sum(dist_matrix[(r[k], r[k+1])] for _, r in routes for k in range(len(r)-1))

## Full Hybrid Solver

In [ ]:
def solve_cvrp_hybrid(depot, customers, N, C, reps=2, verbose=True):
    locations = encode_locations(depot, customers)
    dist_matrix = build_distance_matrix(locations)
    H = len(customers)
    G = compute_num_groups(H, C, N)
    cust_ids = [i for i in locations if i != 0]
    sorted_custs = angular_sort(cust_ids, locations)

    if verbose:
        print(f"  H={H} customers, G={G} group(s), C={C} capacity")

    if G == 1:
        # No cuts needed — skip QAOA
        groups = [sorted_custs]
        resource_info = {"qubits": 0, "gates": 0, "time_s": 0.0}
    else:
        weights = build_cut_weights(sorted_custs, dist_matrix)
        cost_op = build_cost_hamiltonian(weights, G)

        if verbose:
            print(f"  QAOA: {len(weights)} qubits, {G-1} cut(s) needed, reps={reps}")

        probs, opt, resource_info = run_qaoa(cost_op, reps=reps)
        groups = probs_to_groups(probs, sorted_custs, G, weights)

        if verbose:
            print(f"  QAOA: converged={opt.success}, energy={opt.fun:.4f}")

    routes = [(i+1, build_route(g, dist_matrix)) for i, g in enumerate(groups)]
    dist = total_distance(routes, dist_matrix)

    if verbose:
        for i, (_, route) in enumerate(routes):
            print(f"  r{i+1}: {' -> '.join(str(n) for n in route)}")
        print(f"  Total distance: {dist:.4f}")

    return routes, dist, resource_info

## Run All Instances

In [ ]:
depot = (0, 0)

instances = {
    1: {"customers": [(-2,2),(-5,8),(2,3)],                                                                                "N": 2, "C": 5},
    2: {"customers": [(-2,2),(-5,8),(2,3)],                                                                                "N": 2, "C": 2},
    3: {"customers": [(-2,2),(-5,8),(2,3),(5,7),(2,4),(2,-3)],                                                             "N": 3, "C": 2},
    4: {"customers": [(-2,2),(-5,8),(6,3),(4,4),(3,2),(0,2),(-2,3),(-4,3),(2,3),(2,7),(-2,5),(-1,4)], "N": 4, "C": 3},
}

results = {}
resource_table = {}

for inst_num, params in instances.items():
    print(f"{'='*50}")
    print(f"Instance {inst_num}  |  N={params['N']}, C={params['C']}")
    routes, dist, res_info = solve_cvrp_hybrid(depot, params["customers"], params["N"], params["C"])
    results[inst_num] = {"routes": routes, "total_distance": dist}
    resource_table[inst_num] = res_info
    print()

## Resource Usage Table

In [ ]:
print(f"{'Instance':<12} {'Qubits':<10} {'Gates':<10} {'Time (s)':<12}")
print('-'*44)
for inst_num, info in resource_table.items():
    print(f"{inst_num:<12} {info['qubits']:<10} {info['gates']:<10} {info['time_s']:<12.2f}")

## Write Solution Files

In [ ]:
os.makedirs("solutions", exist_ok=True)

for inst_num, res in results.items():
    path = f"solutions/Instance{inst_num}.txt"
    with open(path, "w") as f:
        for i, (_, route) in enumerate(res["routes"], start=1):
            f.write(f"r{i}: " + ", ".join(str(n) for n in route) + "\n")
    print(f"Written {path}")